# 🥈 第 1 阶段：带上下文的对话（Chat Memory）

## 🎯 阶段目标

让模型能够"记住"之前的对话内容：

> **多轮对话 = 历史消息拼接**

这是从"单次调用"到"智能对话"的关键一步。

---

## 📦 1. 加载配置文件

In [1]:
import json
import requests
from pathlib import Path
import time

# 加载配置文件
config_path = Path.cwd() / ".." / "config.json"

with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

MODEL_CONFIG = config["model"]

print("✅ 配置加载成功")
print(f"   提供商：{MODEL_CONFIG['provider']}")
print(f"   模型：{MODEL_CONFIG['name']}")
print(f"   地址：{MODEL_CONFIG['base_url']}")

# 测试连接
print("\n🔍 测试模型连接...")
try:
    test_payload = {
        "model": MODEL_CONFIG['name'],
        "messages": [{"role": "user", "content": "hi"}],
        "max_tokens": 5
    }
    start_time = time.time()
    response = requests.post(
        f"{MODEL_CONFIG['base_url']}/v1/chat/completions",
        json=test_payload,
        timeout=30
    )
    response.raise_for_status()
    elapsed = time.time() - start_time
    print(f"✅ 连接成功！响应时间: {elapsed:.2f}秒")
except requests.exceptions.ConnectionError:
    print(f"❌ 连接失败：无法连接到 {MODEL_CONFIG['base_url']}")
except requests.exceptions.Timeout:
    print("❌ 连接超时")
except Exception as e:
    print(f"❌ 连接失败: {str(e)}")

✅ 配置加载成功
   提供商：vllm
   模型：qwen3.5-122b
   地址：http://36.134.71.54:8000

🔍 测试模型连接...
✅ 连接成功！响应时间: 0.26秒


---

## 🚀 2. 定义带上下文的调用函数

In [2]:
# 初始化对话历史
conversation_history = []

def call_llm_with_history(prompt: str) -> str:
    """
    带上下文的 LLM 调用
    """
    global conversation_history
    
    # 将新的用户输入添加到历史
    conversation_history.append({"role": "user", "content": prompt})
    
    url = f"{MODEL_CONFIG['base_url']}/v1/chat/completions"
    
    payload = {
        "model": MODEL_CONFIG['name'],
        "messages": conversation_history,
        "temperature": MODEL_CONFIG.get('temperature', 0.7),
        "max_tokens": MODEL_CONFIG.get('max_tokens', 100)
    }
    
    try:
        print(f"📤 发送请求... (历史消息数: {len(conversation_history)})")
        start_time = time.time()
        response = requests.post(url, json=payload, timeout=60)
        response.raise_for_status()
        elapsed = time.time() - start_time
        print(f"📥 响应耗时: {elapsed:.2f}秒")
        
        result = response.json()
        answer = result["choices"][0]["message"]["content"]
        
        # 将模型回复添加到历史
        conversation_history.append({"role": "assistant", "content": answer})
        
        return answer
    except requests.exceptions.ConnectionError:
        return f"❌ 错误：无法连接到模型服务 ({MODEL_CONFIG['base_url']})"
    except requests.exceptions.Timeout:
        return "❌ 错误：请求超时"
    except Exception as e:
        return f"❌ 错误：{str(e)}"

print("✅ 带上下文的调用函数定义完成")

✅ 带上下文的调用函数定义完成


---

## 🎮 3. 测试多轮对话

In [3]:
# 测试 1：自我介绍
prompt1 = "我叫张三"
print(f"Q: {prompt1}")
answer1 = call_llm_with_history(prompt1)
print(f"A: {answer1}\n")

Q: 我叫张三
📤 发送请求... (历史消息数: 1)
📥 响应耗时: 31.30秒
A: Thinking Process:

1.  **Analyze the Input:**
    *   Input: "我叫张三" (Wǒ jiào Zhāng Sān)
    *   Translation: "My name is Zhang San."
    *   Context: This is a simple self-introduction.
    *   Intent: The user is introducing themselves.

2.  **Determine the Appropriate Response:**
    *   Acknowledge the name.
    *   Be polite and friendly.
    *   Offer assistance.
    *   Keep it natural and conversational.

3.  **Drafting Responses (Internal Monologue/Trial):**
    *   *Option 1 (Simple):* 你好，张三。(Hello, Zhang San.) - A bit too dry.
    *   *Option 2 (Polite):* 你好，张三！很高兴认识你。(Hello, Zhang San! Nice to meet you.) - Better.
    *   *Option 3 (Helpful):* 你好，张三。我是 AI 助手，有什么可以帮你的吗？(Hello, Zhang San. I'm an AI assistant, is there anything I can help you with?) - Good balance.
    *   *Option 4 (Friendly):* 哇，你好张三！这个名字很经典呢。有什么想聊的吗？(Wow, hello Zhang San! That's a classic name. Anything you want to chat about?) - A bit too casual maybe, but frie

In [4]:
# 测试 2：验证记忆
prompt2 = "我叫什么名字？"
print(f"Q: {prompt2}")
answer2 = call_llm_with_history(prompt2)
print(f"A: {answer2}\n")

Q: 我叫什么名字？
📤 发送请求... (历史消息数: 3)
📥 响应耗时: 13.91秒
A: Thinking Process:

1.  **Analyze the Request:**
    *   User's previous message: "我叫张三" (My name is Zhang San).
    *   User's current message: "我叫什么名字？" (What is my name?).
    *   Context: This is a conversation where the user previously introduced themselves.

2.  **Retrieve Information:**
    *   From the conversation history, the user explicitly stated their name is "张三" (Zhang San).

3.  **Formulate the Answer:**
    *   Direct answer: 你的名字是张三 (Your name is Zhang San).
    *   Tone: Friendly, helpful, acknowledging the previous interaction.
    *   Language: Chinese (matching the user's language).

4.  **Draft the Response:**
    *   "你叫张三呀。" (You are Zhang San.)
    *   "刚才你告诉过我，你叫张三。" (You told me just now, your name is Zhang San.)
    *   "你的名字是张三。" (Your name is Zhang San.)

5.  **Refine the Response:**
    *   Make it natural. "你叫张三呀，刚才你自我介绍过啦。" (You're Zhang San, you introduced yourself just now.)
    *   Keep it concise but

---

## ✅ 本阶段总结

In [5]:
print("📜 当前对话历史：")
for i, msg in enumerate(conversation_history):
    print(f"[{i+1}] {msg['role'].upper()}: {msg['content']}")

📜 当前对话历史：
[1] USER: 我叫张三
[2] ASSISTANT: Thinking Process:

1.  **Analyze the Input:**
    *   Input: "我叫张三" (Wǒ jiào Zhāng Sān)
    *   Translation: "My name is Zhang San."
    *   Context: This is a simple self-introduction.
    *   Intent: The user is introducing themselves.

2.  **Determine the Appropriate Response:**
    *   Acknowledge the name.
    *   Be polite and friendly.
    *   Offer assistance.
    *   Keep it natural and conversational.

3.  **Drafting Responses (Internal Monologue/Trial):**
    *   *Option 1 (Simple):* 你好，张三。(Hello, Zhang San.) - A bit too dry.
    *   *Option 2 (Polite):* 你好，张三！很高兴认识你。(Hello, Zhang San! Nice to meet you.) - Better.
    *   *Option 3 (Helpful):* 你好，张三。我是 AI 助手，有什么可以帮你的吗？(Hello, Zhang San. I'm an AI assistant, is there anything I can help you with?) - Good balance.
    *   *Option 4 (Friendly):* 哇，你好张三！这个名字很经典呢。有什么想聊的吗？(Wow, hello Zhang San! That's a classic name. Anything you want to chat about?) - A bit too casual maybe, but friendly.

